#The "Hello World" of Reinforcement Learning: 1D Treasure Hunt

Imagine a 1D hallway with 5 rooms:[Trap] [Empty] [Start] [Empty] [Treasure]

Our AI agent will be dropped into the Start room (State 2). It does not know the rules of the hallway. It only knows it can take two actions: Move Left (0) or Move Right (1).

The Reward System:
*  Falling into the Trap (State 0) gives a reward of -1.
*  Finding the Treasure (State 4) gives a reward of +1.
*  Moving to an empty room gives a reward of 0.

The Goal: We will use the Bellman Equation to train a Q-table. The table will start completely blank (all zeros), but as the agent stumbles around the hallway, it will record the expected future reward for every possible action. Run the code below to watch the agent learn!

In [ ]:
import numpy as np
import random

# Hyperparameters
alpha = 0.5      # Learning rate
gamma = 0.9      # Discount factor
epsilon = 0.2    # Exploration rate
episodes = 50

# The Q-Table: 5 states (rows), 2 actions (columns: 0=Left, 1=Right)
q_table = np.zeros((5, 2))

# Environment definitions
START_STATE = 2
TRAP_STATE = 0
TREASURE_STATE = 4

for episode in range(episodes):
    state = START_STATE
    done = False

    while not done:
        # Epsilon-Greedy Action Selection
        if random.uniform(0, 1) < epsilon:
            action = random.choice([0, 1]) # Explore: Random Left or Right
        else:
            action = np.argmax(q_table[state]) # Exploit: Best known action

        # Take the action and observe the new state
        if action == 0: # Left
            next_state = max(0, state - 1)
        else:           # Right
            next_state = min(4, state + 1)

        # Observe the reward
        if next_state == TREASURE_STATE:
            reward = 1
            done = True
        elif next_state == TRAP_STATE:
            reward = -1
            done = True
        else:
            reward = 0

        # The Bellman Equation Update
        current_q = q_table[state, action]
        max_next_q = np.max(q_table[next_state])

        q_table[state, action] = current_q + alpha * (reward + gamma * max_next_q - current_q)

        state = next_state

print("Training Complete! The final Q-Table:")
print("State | Q(Left) | Q(Right)")
for s in range(5):
    print(f"  {s}   |  {q_table[s, 0]:.3f}  |  {q_table[s, 1]:.3f}")

Training Complete! The final Q-Table:
State | Q(Left) | Q(Right)
  0   |  0.000  |  0.000
  1   |  -0.750  |  0.777
  2   |  0.564  |  0.900
  3   |  0.784  |  1.000
  4   |  0.000  |  0.000


# Decoding the Q-TableThe printed grid is the agent's Q-table
*  Decisions: The agent always picks the highest number in its current row (e.g., in State 2, Right > Left).
*  Danger: State 1 Left is -1.000. The agent learned to avoid the trap.
*  Looking Ahead: Notice moving Right from States 1, 2, and 3: 0.810 $\rightarrow$ 0.900 $\rightarrow$ 1.000. The +1 treasure reward trickles backwards through the empty rooms (decaying by gamma = 0.9), creating a numerical trail to follow.
*  Terminal Zeros: The Trap (0) and Treasure (4) rows remain 0.000 because the game immediately ends upon entering them.